# v1 Pre-Flight Gate — Literature CSV data review

**Purpose.** Verify every digitized literature CSV under
`data/literature/{BTO,STO}/` against the v1 inventory before any v1
module work begins. Per the data-review checklist: *no new module files until this
notebook runs clean*.

This notebook does **no physics**. It loads every CSV, asserts the headline
numbers recorded during v1, plots five raw-data review figures (no fits
except the LST R² which is informational), summarizes the STO Γ availability,
tags phases on every BTO Λ² source, and emits a pass/fail checklist at the end.

If any assertion fails, any plot shows something the v1 inventory did not
predict, or the checklist contains a ✗, **stop and report** before continuing.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # editable-install fallback

DATA_RAW = PROJECT_ROOT / "data" / "raw"
LIT = PROJECT_ROOT / "data" / "literature"
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

CM1_TO_MEV = 0.1239841984  # 1 cm^-1 in meV

from fekappa.plots import set_publication_style
set_publication_style()

# Result trackers used by the checklist cell at the end.
CHECKLIST: list[tuple[str, bool, str]] = []  # (item, passed, note)
ASSERTIONS: list[tuple[str, bool, str]] = []  # (label, passed, detail)


def record_assertion(label: str, passed: bool, detail: str = "") -> None:
    """Print one PASS/FAIL line and append to ASSERTIONS for the checklist."""
    mark = "PASS" if passed else "FAIL"
    print(f"  [{mark}] {label}" + (f"  ({detail})" if detail else ""))
    ASSERTIONS.append((label, passed, detail))


def record_checklist(item: str, passed: bool, note: str = "") -> None:
    CHECKLIST.append((item, passed, note))


def isclose(a: float, b: float, abs_tol: float = 1e-2, rel_tol: float = 0.02) -> bool:
    return abs(a - b) <= max(abs_tol, rel_tol * abs(b))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LIT:        ", LIT)
print("FIG_DIR:    ", FIG_DIR)


## 1. CSV inventory

For each file under `data/literature/{BTO,STO}/**/*.csv`: relative path, shape,
column dtypes, and first 3 rows.

In [ ]:
csv_paths = sorted((LIT).rglob("*.csv"))
print(f"Found {len(csv_paths)} CSV files under data/literature/.\n")
loaded: dict[str, pd.DataFrame] = {}
load_failures: list[tuple[str, str]] = []

for p in csv_paths:
    rel = str(p.relative_to(LIT))
    try:
        df = pd.read_csv(p)
    except Exception as exc:  # noqa: BLE001
        load_failures.append((rel, repr(exc)))
        print(f"--- {rel}  (LOAD ERROR: {exc!r})")
        continue
    loaded[rel] = df
    print(f"--- {rel}")
    print(f"    shape: {df.shape}")
    print("    dtypes:")
    for col, dt in df.dtypes.items():
        print(f"      {col}: {dt}")
    print("    head(3):")
    if df.empty:
        print("      (empty)")
    else:
        head_str = df.head(3).to_string(index=False).replace("\n", "\n      ")
        print(f"      {head_str}")
    print()

record_checklist(
    "All CSVs loaded without error",
    not load_failures,
    f"{len(loaded)} loaded, {len(load_failures)} failed",
)


## 2. Headline number assertions

Hardcoded expected values from the v1 inventory. Each line prints
PASS/FAIL and is accumulated into the pre-flight checklist below.

In [ ]:
# --- BTO Ehsan anchors ---
df = loaded["BTO/ehsan_2021/anchors.csv"]
anchors_by_T = {int(float(r["T_K"])): r for _, r in df.iterrows()}

a0 = anchors_by_T[0]
a500 = anchors_by_T[500]
record_assertion("BTO Ehsan anchor T=0 K: Omega_sq_meV2 == -424",
                 isclose(float(a0["Omega_sq_meV2"]), -424.0, abs_tol=0.5),
                 detail=f"got {a0['Omega_sq_meV2']}")
record_assertion("BTO Ehsan anchor T=500 K: Omega_meV == 21.08",
                 isclose(float(a500["Omega_meV"]), 21.08, abs_tol=0.02),
                 detail=f"got {a500['Omega_meV']}")
record_assertion("BTO Ehsan anchor T=500 K: Omega_sq_meV2 == 444.4",
                 isclose(float(a500["Omega_sq_meV2"]), 444.4, abs_tol=0.5),
                 detail=f"got {a500['Omega_sq_meV2']}")

# --- BTO Shirane Lambda^2 central ---
df = loaded["BTO/shirane_1970/Lambda_sq.csv"]
lambda_sq_central = float(df.iloc[0]["Lambda_sq_meV2_A2"])
record_assertion("BTO Shirane Lambda^2 central == 5100 meV^2 A^2",
                 isclose(lambda_sq_central, 5100.0, abs_tol=1.0),
                 detail=f"got {lambda_sq_central}")

# --- BTO Ehsan structural (cubic) ---
df = loaded["BTO/ehsan_2021/structural_properties.csv"]
cubic = df[df["phase"] == "cubic"].iloc[0]
record_assertion("BTO Ehsan cubic a == 4.013 A",
                 isclose(float(cubic["a_angstrom"]), 4.013, abs_tol=0.001),
                 detail=f"got {cubic['a_angstrom']}")
record_assertion("BTO Ehsan cubic V_cell == 64.63 A^3",
                 isclose(float(cubic["V_cell_A3"]), 64.63, abs_tol=0.02),
                 detail=f"got {cubic['V_cell_A3']}")

# --- BTO Tadano 2022 transition temperatures (cubic-tetra) ---
df = loaded["BTO/tadano_2022/transition_temperatures.csv"]
ctt = df[df["transition"] == "cubic-tetra"].iloc[0]
record_assertion("BTO Tadano 2022 Tc(cubic-tetra) calc raw == 606 K",
                 isclose(float(ctt["T_c_calc_K"]), 606.0, abs_tol=1.0),
                 detail=f"got {ctt['T_c_calc_K']}")
record_assertion("BTO Tadano 2022 Tc(cubic-tetra) exp == 390 K",
                 isclose(float(ctt["T_c_exp_K"]), 390.0, abs_tol=1.0),
                 detail=f"got {ctt['T_c_exp_K']}")
# Lattice-corrected 511 K is body-text only (extraction_log), not in any CSV.
record_assertion("BTO Tadano 2022 Tc(cubic-tetra) lattice-corrected == 511 K  "
                 "[body text per extraction_log; informational]",
                 True, detail="recorded in extraction_log.md, not in any CSV")

# --- BTO Tadano 2022 kappa(T) endpoints ---
df = loaded["BTO/tadano_2022/kappa_total_T.csv"]
calc = df[df["method"] == "Tadano_calc"].sort_values("T_K")
k200 = float(calc[calc["T_K"] == 200]["kappa_W_per_m_K"].iloc[0])
k800 = float(calc[calc["T_K"] == 800]["kappa_W_per_m_K"].iloc[0])
record_assertion("BTO Tadano 2022 kappa @200 K Tadano_calc ~ 10.3",
                 isclose(k200, 10.3, abs_tol=0.1),
                 detail=f"got {k200}")
record_assertion("BTO Tadano 2022 kappa @800 K Tadano_calc ~ 5.4",
                 isclose(k800, 5.4, abs_tol=0.1),
                 detail=f"got {k800}")

# --- STO Tadano 2015 scph_omega_T: exactly the 5 methods of the checklist ---
df = loaded["STO/tadano_2015/scph_omega_T.csv"]
methods = sorted(df["method"].unique())
expected_methods = sorted(["SCPH_with_PM", "SCPH_without_PM", "TDEP",
                           "exp_Servoin", "exp_YamadaShirane"])
record_assertion("STO Tadano 2015 scph_omega_T has exactly the 5 expected methods",
                 methods == expected_methods,
                 detail=f"got {methods}")

# --- STO Verdi 2023 sscha_omega_T shift_meV ---
df = loaded["STO/verdi_2023/sscha_omega_T.csv"]
mlff = df[df["method"] == "MLFF_SSCHA"]
unique_shifts = sorted(mlff["shift_meV"].unique())
record_assertion("STO Verdi sscha_omega_T MLFF_SSCHA shift_meV == 4.93 (single value)",
                 unique_shifts == [4.93],
                 detail=f"got {unique_shifts}")

# --- STO Verdi inverse_dielectric_T ---
df = loaded["STO/verdi_2023/inverse_dielectric_T.csv"]
mlff = df[df["method"] == "MLFF_SSCHA"].sort_values("T_K")
inv_eps_10 = float(mlff[mlff["T_K"] == 10]["inverse_epsilon"].iloc[0])
inv_eps_300 = float(mlff[mlff["T_K"] == 300]["inverse_epsilon"].iloc[0])
record_assertion("STO Verdi 1/eps(10 K) MLFF_SSCHA ~ 0.0006",
                 isclose(inv_eps_10, 0.0006, abs_tol=5e-5),
                 detail=f"got {inv_eps_10}")
record_assertion("STO Verdi 1/eps(300 K) MLFF_SSCHA ~ 0.0243",
                 isclose(inv_eps_300, 0.0243, abs_tol=5e-4),
                 detail=f"got {inv_eps_300}")

n_pass = sum(1 for _, p, _ in ASSERTIONS if p)
n_total = len(ASSERTIONS)
print(f"\nSummary: {n_pass}/{n_total} assertions passed.")
record_checklist("All headline number assertions PASS", n_pass == n_total,
                 f"{n_pass}/{n_total}")


## 2b. Wave-1 arXiv source assertions (Fauqué 2022 / Schmidt-Spaldin 2025 / Meier 2022)


In [ ]:
_w1_start = len(ASSERTIONS)

# --- Fauqué 2022 Fig. 1(d): FE soft mode = Γ-point TO; NO Γ reported ---
df = loaded["STO/fauque_2022/dho_fits_T.csv"]
record_assertion("STO Fauqué gamma_meV entirely NaN (no DHO Γ in source)",
                 bool(df["gamma_meV"].isna().all()),
                 detail="locks the no-Γ finding; η stays a bracket for STO")
record_assertion("STO Fauqué gamma_uncertainty_meV entirely NaN",
                 bool(df["gamma_uncertainty_meV"].isna().all()))
to_exp = df[(df["mode"] == "TO") & (df["method"] == "experimental")].sort_values("T_K")
fq_lo = float(to_exp[to_exp["T_K"] == 4]["omega_meV"].iloc[0])
fq_hi = float(to_exp[to_exp["T_K"] == 1400]["omega_meV"].iloc[0])
record_assertion("STO Fauqué Γ-TO exp low-T Ω(4 K) ~ 2.0 meV",
                 isclose(fq_lo, 2.0, abs_tol=0.05), detail=f"got {fq_lo}")
record_assertion("STO Fauqué Γ-TO exp high-T Ω(1400 K) ~ 17.3 meV",
                 isclose(fq_hi, 17.3, abs_tol=0.05), detail=f"got {fq_hi}")
record_assertion("STO Fauqué non-FE modes (R-AFD, M) tagged distinctly from TO",
                 set(df["mode"].unique()) == {"TO", "AFD", "M_mode"},
                 detail=f"modes={sorted(df['mode'].unique())}")

# --- Schmidt-Spaldin 2025 Fig. 3 (top left): 16O/18O Ω(T); imaginary convention ---
df = loaded["STO/schmidt_2025/sscha_omega_T.csv"]
record_assertion("STO Schmidt has both isotopes ST16O + ST18O",
                 {"ST16O", "ST18O"} <= set(df["isotope"].unique()),
                 detail=f"isotopes={sorted(df['isotope'].unique())}")
st18_0 = df[(df["isotope"] == "ST18O") & (df["method"] == "SSCHA") & (df["T_K"] == 0)]
st18_0_om = float(st18_0["Omega_meV"].iloc[0])
record_assertion("STO Schmidt ST18O SSCHA Ω(0 K) imaginary (negative)",
                 st18_0_om < 0,
                 detail=f"got {st18_0_om} meV (kernel skips Ω²<0)")

# --- Meier 2022 Fig. 3(a): KTaO3 Ω(T) plateau + endpoint; lattice lock ---
df = loaded["KTO/meier_2022/omega_T.csv"]
calc = df[df["method"] == "Calculated"]
o0 = float(calc[calc["T_K"] == 0]["Omega_meV"].iloc[0])
o10 = float(calc[calc["T_K"] == 10]["Omega_meV"].iloc[0])
record_assertion("KTaO3 Meier quantum-paraelectric plateau Ω(0 K)==Ω(10 K)",
                 isclose(o0, o10, abs_tol=1e-6), detail=f"Ω(0)={o0}, Ω(10)={o10} meV")
o300 = float(calc[calc["T_K"] == 300]["Omega_meV"].iloc[0])
record_assertion("KTaO3 Meier Ω(300 K) calc ~ 10.86 meV",
                 isclose(o300, 10.86, abs_tol=0.05), detail=f"got {o300}")
V_meier = 4.007 ** 3
record_assertion("KTaO3 lattice lock a=4.007 Å -> V_cell=64.34 Å³ (Meier calc)",
                 isclose(V_meier, 64.34, abs_tol=0.02),
                 detail=f"V={V_meier:.2f}; exp a=3.990->{3.990**3:.2f}, ΔV<1.5%")
dd = loaded["KTO/meier_2022/dispersion_T.csv"]
soft = dd[dd["branch"] == "Soft_Optical"]
record_assertion("KTaO3 Meier soft mode is Γ-only (Λ²_KTO needs another source)",
                 set(soft["q_label"].unique()) == {"Gamma"},
                 detail=f"soft-branch q-points={sorted(soft['q_label'].unique())}")

_w1 = ASSERTIONS[_w1_start:]
n_pass_w1 = sum(1 for _, p, _ in _w1 if p)
print(f"\nWave-1 (Fauqué/Schmidt/Meier): {n_pass_w1}/{len(_w1)} assertions passed.")
record_checklist("Wave-1 arXiv source assertions PASS (Fauqué/Schmidt/Meier)",
                 n_pass_w1 == len(_w1), f"{n_pass_w1}/{len(_w1)}")


## 2c. Wave-2 library source assertions (Harada-Axe-Shirane 1971 + Yamada-Shirane 1969)


In [ ]:
_w2_start = len(ASSERTIONS)

# --- HAS 1971 (cubic-phase BTO): overdamped soft TO + cubic Lambda^2 ---
df = loaded["BTO/harada_axe_shirane_1971/mode_parameters.csv"]
record_assertion("HAS 1971 mode_parameters all CUBIC phase (Table-I 'K' is a misprint; figs say C)",
                 (df["phase"] == "cubic").all(),
                 detail=f"T_K={sorted(df['T_K'].unique())} = 150/250/450 C")
to = df[df["mode"] == "TO"]
eta = to["gamma_meV"].astype(float) / to["omega_meV"].astype(float)
record_assertion("HAS 1971 soft TO OVERDAMPED: Gamma_TO >= 2 Omega_TO at all (q, T)",
                 bool((to["gamma_meV"] >= 2 * to["omega_meV"]).all()),
                 detail=f"eta range {eta.min():.2f}-{eta.max():.2f} (paper: 2.3)")
hi = to[to["T_K"] == 723].iloc[0]
record_assertion("HAS 1971 Gamma_TO=34, Omega_TO=15 meV (q=0.2, 450C): real overdamped DHO linewidth",
                 isclose(float(hi["gamma_meV"]), 34.0, abs_tol=0.5)
                 and isclose(float(hi["omega_meV"]), 15.0, abs_tol=0.5),
                 detail="DHO damping (Eq. 2), not a FWHM/HWHM artifact")
cd = loaded["BTO/harada_axe_shirane_1971/cubic_dispersion.csv"]
to_q = sorted(cd[cd["branch"] == "TO"]["q_inv_angstrom"].astype(float).unique())
record_assertion("HAS 1971 cubic soft-TO at small-q (usable for Lambda^2, not zone-boundary-only)",
                 max(to_q) < 0.5,
                 detail=f"q={to_q} A^-1 (zeta<=0.3); cf. Ehsan zone-boundary-only")
ls = loaded["BTO/harada_axe_shirane_1971/Lambda_sq.csv"].iloc[0]
record_assertion("HAS 1971 cubic Lambda^2(100)=972+/-50 meV^2 A^2 (first CUBIC-phase BTO Lambda^2)",
                 isclose(float(ls["Lambda_sq_meV2_A2"]), 972.0, abs_tol=1.0),
                 detail="resolves NB 05b finding that all prior BTO Lambda^2 were tetragonal")
ana = loaded["BTO/harada_axe_shirane_1971/anisotropy.csv"]
record_assertion("HAS 1971 Fig. 7 signed angles within (-90, 90) about [100]=0 deg",
                 bool(ana["angle_deg"].between(-90, 90).all() and (ana["angle_deg"] == 0).any()),
                 detail=f"range [{ana['angle_deg'].min()}, {ana['angle_deg'].max()}] deg")

# --- YS 1969 (cubic STO): experimental Gamma(T), underdamped ---
dg = loaded["STO/yamada_shirane_1969/gamma_T.csv"]
g20 = float(dg[dg["T_K"] == 20]["gamma_meV"].iloc[0])
g250 = float(dg[dg["T_K"] == 250]["gamma_meV"].iloc[0])
record_assertion("YS 1969 Fig. 6 Gamma(T): 0.24 @20K -> 0.84 meV @250K (verified vs figure)",
                 isclose(g20, 0.24, abs_tol=0.01) and isclose(g250, 0.84, abs_tol=0.01),
                 detail="canonical low-T exp STO linewidth; enables eta_fit (NB 08)")
do = loaded["STO/yamada_shirane_1969/omega_T.csv"]
o250 = float(do[(do["condition_direction"] == "q_0_TO_soft") & (do["T_K"] == 250)]["omega_meV"].iloc[0])
record_assertion("STO soft mode UNDERDAMPED (eta~0.09 @250K) -- contrast cubic BTO eta~2.3",
                 (g250 / o250) < 0.2,
                 detail=f"eta_STO={g250 / o250:.3f} vs eta_BTO~2.3 (regime difference is physical)")

_w2 = ASSERTIONS[_w2_start:]
n_pass_w2 = sum(1 for _, p, _ in _w2 if p)
print(f"\nWave-2 (HAS 1971 + YS 1969): {n_pass_w2}/{len(_w2)} assertions passed.")
record_checklist("Wave-2 library source assertions PASS (HAS 1971 + YS 1969)",
                 n_pass_w2 == len(_w2), f"{n_pass_w2}/{len(_w2)}")


## 2d. KTaO$_3$-wave assertions (Ranalli 2023; two-source triangulation)


In [ ]:
_w3_start = len(ASSERTIONS)
dr = loaded["KTO/ranalli_2023/omega_T.csv"]
record_assertion("Ranalli omega_T methods present (MLFF+SSCHA, expt, fit_paraelectric)",
                 {"MLFF+SSCHA", "expt", "fit_paraelectric"} <= set(dr["method"].unique()),
                 detail=f"methods={sorted(dr['method'].unique())}; native cm⁻¹→meV")
ml = dr[dr["method"] == "MLFF+SSCHA"]
record_assertion("Ranalli MLFF+SSCHA shifted −4.30 meV (=34.71 cm⁻¹; Verdi-family convention)",
                 bool((ml["shift_meV"] == -4.30).all()),
                 detail="captured as plotted; recover unshifted via Ω − shift_meV")
record_assertion("Ranalli QPE low-T plateau Ω(5 K) == Ω(20 K)",
                 isclose(float(ml[ml["T_K"] == 5]["Omega_meV"].iloc[0]),
                         float(ml[ml["T_K"] == 20]["Omega_meV"].iloc[0]), abs_tol=1e-6),
                 detail=f"plateau ≈ {float(ml[ml['T_K'] == 5]['Omega_meV'].iloc[0]):.2f} meV (≈20 cm⁻¹)")
dm = loaded["KTO/meier_2022/omega_T.csv"]
rex = float(dr[(dr["method"] == "expt") & (dr["T_K"] == 300)]["Omega_meV"].iloc[0])
mex = float(dm[(dm["method"] == "Shirane_exp") & (dm["T_K"] == 300)]["Omega_meV"].iloc[0])
record_assertion("KTaO₃ INS exp anchors AGREE across sources @300 K (Ranalli vs Meier)",
                 abs(rex - mex) < 1.0,
                 detail=f"Ranalli {rex:.2f} vs Meier {mex:.2f} meV (Δ={abs(rex - mex):.2f})")
rc = float(dr[(dr["method"] == "MLFF+SSCHA") & (dr["T_K"] == 300)]["Omega_meV"].iloc[0])
mc = float(dm[(dm["method"] == "Calculated") & (dm["T_K"] == 300)]["Omega_meV"].iloc[0])
record_assertion("KTaO₃ computed-curve spread bounded @300 K (shift/method, NOT a digitization error)",
                 abs(rc - mc) < 3.0,
                 detail=f"Ranalli {rc:.2f} vs Meier {mc:.2f} meV (Δ={abs(rc - mc):.2f}; shift-convention spread)")
_w3 = ASSERTIONS[_w3_start:]
n3 = sum(1 for _, p, _ in _w3 if p)
print(f"\nKTaO₃ wave (Ranalli 2023): {n3}/{len(_w3)} assertions passed.")
record_checklist("KTaO₃-wave assertions PASS (Ranalli 2023 + 2-source consistency)",
                 n3 == len(_w3), f"{n3}/{len(_w3)}")


## 2e. Vogt-wave assertions (Vogt 1995 experimental Ω+γ; γ-convention; YS-vs-Vogt)


In [ ]:
_w4_start = len(ASSERTIONS)
ko = loaded["KTO/vogt_1995/omega_T.csv"]; kg = loaded["KTO/vogt_1995/gamma_T.csv"]
so = loaded["STO/vogt_1995/omega_T.csv"]; sg = loaded["STO/vogt_1995/gamma_T.csv"]
record_assertion("Vogt shift_meV = 0 for all rows (experimental, no shift)",
                 bool((ko["shift_meV"] == 0).all() and (so["shift_meV"] == 0).all()))
record_assertion("Vogt STO carries phase_mode; cubic_F1u filterable & >=110 K (kernel eta)",
                 bool(("phase_mode" in so.columns) and ("phase_mode" in sg.columns)
                      and so[so["phase_mode"] == "cubic_F1u"]["T_K"].min() >= 110),
                 detail="tetragonal A2u/Eu < 105 K excluded from cubic eta")
record_assertion("Two-variant gamma schema intact (KTaO3-Vogt 3-col, STO-Vogt 4-col; not reconciled)",
                 (list(kg.columns) == ["T_K", "gamma_meV", "gamma_uncertainty_meV"])
                 and ("phase_mode" in sg.columns))
# STO dense (non-round T): cubic_F1u high-T endpoints (transition_merged excluded).
so_f1u = so[so["phase_mode"] == "cubic_F1u"].sort_values("T_K")
sg_f1u = sg[sg["phase_mode"] == "cubic_F1u"].sort_values("T_K")
sO = float(so_f1u["Omega_meV"].iloc[-1])             # ~10.96 meV @ 299 K
sG = float(sg_f1u["gamma_meV"].iloc[-1])             # ~2.40 meV @ 298 K (dense)
# KTaO3 dense WebPlotDigitizer extraction (non-round T) -> use high-T endpoints.
ko95 = ko[ko["method"] == "exp_HyperRaman_Vogt1995"].sort_values("T_K")
kO = float(ko95["Omega_meV"].iloc[-1])                   # ~10.15 meV @ 294 K (1995 curve)
kG = float(kg.sort_values("T_K")["gamma_meV"].iloc[-1])  # ~2.57 meV @ 296 K
eS = 0.5 * sG / sO; eK = 0.5 * kG / kO
record_assertion("gamma-convention recorded: plotted Vogt gamma=FWHM -> Gamma_HWHM=gamma/2 (factor-of-2 noted)",
                 True, detail=f"eta_STO={eS:.3f}, eta_KTO={eK:.3f} [canonical Gamma_HWHM/Omega, gamma as FWHM]; x2 to ~{2*eS:.2f}/{2*eK:.2f} if gamma were HWHM")
record_assertion("QPE eta CLUSTER: STO ~ KTaO3, both underdamped (eta<0.5), far from BTO ~2.3",
                 eS < 0.5 and eK < 0.5 and abs(eS - eK) < 0.05,
                 detail=f"|eta_STO-eta_KTO|={abs(eS - eK):.3f}; empty interior to BTO 2.3")
ys = loaded["STO/yamada_shirane_1969/gamma_T.csv"]
ys250 = float(ys[ys["T_K"] == 250]["gamma_meV"].iloc[0])
sgf = sg[sg["phase_mode"] == "cubic_F1u"].copy()
vg250 = float(sgf.iloc[(sgf["T_K"] - 250).abs().argmin()]["gamma_meV"])  # nearest 250 (dense)
record_assertion("YS-vs-Vogt STO damping spread documented (NOT reconciled to equality)",
                 (vg250 / ys250) > 2.0,
                 detail=f"raw gamma ratio {vg250 / ys250:.2f} (dense; was ~3.3 round); ~{0.5 * vg250 / ys250:.1f}x residual after /2")
record_assertion("Vogt KTaO3 high-T Omega tie-breaks Meier-Ranalli toward higher (Meier-like) freq",
                 abs(kO - 10.86) < abs(9.05 - 10.86),
                 detail=f"Vogt {kO:.2f} (294 K, dense) vs Meier 10.86 vs Ranalli-shifted 9.05 (qualitatively holds; less emphatic than old 10.48)")
# Re-verification of the corrected KTaO3 gamma tail (vs Fig. 5).
record_assertion("KTaO3 Vogt = dense WPD extraction, figure-faithful (high-T endpoint 2.57 meV=20.7 cm-1; boundary 7->9.7 cm-1 smooth rise, no scaling artifact)",
                 isclose(kG, 2.57, abs_tol=0.02),
                 detail="raw (never scaled): no flat-junction artifact; 296/298 K double-pick dropped; mid-T scatter + 11.5 K sub-bar dip retained as faithful")
# Uncertainty convention: sub-60 K quadrature 0.17 = sqrt(0.12^2+0.12^2); >=60 K = 0.12; ALL Vogt files.
def _unc_ok(df, col):
    # The 0.12 meV (±1 cm⁻¹) / sub-60 K quadrature (0.17) convention applies to
    # the Vogt-1995 Fig. 5/6 data; the 1988 Table III cross-ref point legitimately
    # keeps its own ±0.37 meV, so exclude non-1995 method rows.
    d = df[df["method"].str.contains("1995")] if "method" in df.columns else df
    s60 = d[d["T_K"] < 60][col].astype(float)
    g60 = d[d["T_K"] >= 60][col].astype(float)
    return bool((abs(s60 - 0.17) < 1e-6).all() and (abs(g60 - 0.12) < 1e-6).all())
record_assertion("Sub-60 K unc = 0.17 meV (stat 0.12 + sys 0.12 quadrature), >=60 K = 0.12; ALL Vogt files",
                 _unc_ok(kg, "gamma_uncertainty_meV") and _unc_ok(sg, "gamma_uncertainty_meV")
                 and _unc_ok(ko, "Omega_uncertainty_meV") and _unc_ok(so, "Omega_uncertainty_meV"),
                 detail="Vogt p.8051 defect-induced sub-60 K systematic; 60 K boundary = 'below about 60 K'")
record_assertion("Cross-file consistency: STO Vogt uses same 0.12/0.17 convention as KTaO3 (no 0.05/0.08 desync)",
                 set(np.round(sg["gamma_uncertainty_meV"].astype(float), 2)) <= {0.12, 0.17}
                 and set(np.round(so["Omega_uncertainty_meV"].astype(float), 2)) <= {0.12, 0.17})
# Flag-2 / check-2: STO sub-105 K DAMPING is ONE shared-gamma branch (Vogt p.8051),
# NOT two diverging A2u/Eu branches. The dense gamma represents it as a single
# sub-105 K branch (no separate Eu row); the FREQUENCY does split (omega has both
# A2u and Eu), but per Vogt's fit the damping is shared.
_sub105 = sg[sg["T_K"] < 105]
record_assertion("Flag-2 / check-2: STO sub-105 K damping is a SINGLE shared-gamma branch (no two diverging branches)",
                 _sub105["phase_mode"].nunique() == 1,
                 detail=f"sub-105K gamma tag(s)={sorted(_sub105['phase_mode'].unique())} -> one curve (shared A2u=Eu gamma, Vogt p.8051)")
record_assertion("STO dense WPD: 3 sub-105K points re-tagged transition_merged -> cubic_F1u starts >=110K (both files); uncertainty boundary 60K",
                 bool(so_f1u["T_K"].min() >= 110 and sg_f1u["T_K"].min() >= 110),
                 detail=f"eta_STO={0.5*sG/sO:.3f} from dense gamma (marker-center; 3rd figure read ~21-22+/-2 cm-1, NOT corrected toward round 24.5); clusters w/ KTaO3")
_w4 = ASSERTIONS[_w4_start:]
n4 = sum(1 for _, p, _ in _w4 if p)
print(f"\nVogt wave (Vogt 1995 STO + KTaO3): {n4}/{len(_w4)} assertions passed.")
record_checklist("Vogt-wave assertions PASS (experimental Omega+gamma both QPEs; gamma-convention; YS-vs-Vogt)",
                 n4 == len(_w4), f"{n4}/{len(_w4)}")


## 3. Figure A — BTO $\Omega^2(T)$ (Ehsan anchors)

Two points: $T=0$ K bare (negative, imaginary) and $T=500$ K thermalized
(positive). No interpolation, no fit. The shaded region $\Omega^2<0$ marks the
unstable / non-quasiparticle regime.

In [ ]:
df = loaded["BTO/ehsan_2021/anchors.csv"]
T = df["T_K"].astype(float).to_numpy()
O2 = df["Omega_sq_meV2"].astype(float).to_numpy()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.axhspan(-600, 0, color="0.85", alpha=0.5, label=r"$\Omega^2<0$ (unstable)")
ax.axhline(0.0, color="k", lw=0.8)
ax.scatter(T, O2, color="tab:red", s=80, zorder=5, label="Ehsan 2021 anchors")
for ti, oi in zip(T, O2):
    label = f"  T={ti:.0f} K\n  $\\Omega^2$={oi:.0f} meV$^2$"
    ax.annotate(label, (ti, oi), textcoords="offset points", xytext=(8, -6),
                fontsize=9, va="top")
ax.set_xlim(-30, 620)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\Omega^2$ (meV$^2$)")
ax.set_title("Figure A — BTO soft-mode $\\Omega^2(T)$ (Ehsan 2 anchors)")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
plt.show()
record_checklist("Figure A renders", True)


## 4. Figure B — BTO $\Lambda^2$ source overlay (no fits)

Four sources of the BTO soft-mode q-dispersion overlaid on $\Omega^2$ vs $q^2$:

1. **Shirane 1970 prior fit** ($\Lambda^2 = 5100$ meV$^2$Å$^2$, central lock):
   dashed reference line $\Omega^2 = 5100\,q^2$ through the origin.
2. **Shirane Fig. 6, Δ branch (e‖[100])**: 295 K, tetragonal. Small-q TO points.
3. **Shirane Fig. 3, Σ branch (e‖[110])**: 295 K, tetragonal. Small-q TO points.
4. **Ehsan Fig. 2 finite-q points**: 0 K nonthermalized + 500 K thermalized. *Zone-boundary* points (X, M, R), not small-q.

**v1 fix (option 3):** the
`cubic_dispersion.csv` `q_inv_angstrom` column has been corrected in place
from reduced units to Å⁻¹ (multiplied by $2\pi/4.013 \approx 1.5657$;
Ehsan Table I cubic $a$). The Ehsan finite-q points sit at zone boundary,
*outside* the small-q regime where $\Omega^2(q) = \Omega_0^2 + \Lambda^2 q^2$ is
valid, so they are **NOT used for the $\Lambda^2$ refit in NB 06**. The
apparent "negative slope" $\Omega(X) < \Omega(\Gamma)$ at 500 K thermalized
is interpreted as SCP soft-branch anti-crossing with another mode along
Γ→X, not as $\Lambda^2 < 0$. The Ehsan dispersion retains standalone value
as a shape illustration of the SCP scheme; only Shirane Σ and Δ small-q TO
points feed the BTO $\Lambda^2$ refit (both 295 K tetragonal — the cubic-
phase kernel inherits the phase-shift uncertainty, absorbed by the ±20%
sensitivity envelope).

In [ ]:
# Shirane cubic_dispersion: TO branch only, separate Σ (Fig 3) vs Δ (Fig 6).
df_sh = loaded["BTO/shirane_1970/cubic_dispersion.csv"]
sh_to = df_sh[df_sh["branch"] == "TO"].copy()
sh_to["q_sq"] = sh_to["q_inv_angstrom"].astype(float) ** 2
sh_to["omega_sq_meV2"] = sh_to["omega_meV"].astype(float) ** 2
sh_sigma = sh_to[sh_to["source_figure"] == "Fig_3"]
sh_delta = sh_to[sh_to["source_figure"] == "Fig_6"]

# Ehsan cubic_dispersion: soft-mode points. q_inv_angstrom is now in A^-1
# after the v1 pre-flight unit conversion (see extraction_log.md).
df_eh = loaded["BTO/ehsan_2021/cubic_dispersion.csv"]
eh_soft = df_eh[df_eh["branch_label"] == "Ferroelectric_Soft_Mode"].copy()
eh_soft["q_sq"] = eh_soft["q_inv_angstrom"].astype(float) ** 2
eh_soft["omega_meV"] = eh_soft["omega_cm_inv"].astype(float) * CM1_TO_MEV
eh_soft["omega_sq_meV2"] = np.sign(eh_soft["omega_meV"]) * eh_soft["omega_meV"] ** 2
eh_0K = eh_soft[eh_soft["condition"] == "nonthermalized"]
eh_500K = eh_soft[eh_soft["condition"] == "thermalized"]

fig, ax = plt.subplots(figsize=(8, 5.2))
ax.axhline(0.0, color="0.6", lw=0.6)

q_ref = np.linspace(0, 1.8, 200)
ax.plot(q_ref**2, 5100 * q_ref**2, "k--", lw=1.0,
        label=r"Shirane prior fit  $\Omega^2 = 5100\,q^2$")

ax.scatter(sh_delta["q_sq"], sh_delta["omega_sq_meV2"], marker="o",
           color="tab:blue", s=55,
           label=r"Shirane Δ (Fig. 6, e‖[100], 295 K tet., small-q)")
ax.scatter(sh_sigma["q_sq"], sh_sigma["omega_sq_meV2"], marker="s",
           color="tab:green", s=55,
           label=r"Shirane Σ (Fig. 3, e‖[110], 295 K tet., small-q)")
ax.scatter(eh_0K["q_sq"], eh_0K["omega_sq_meV2"], marker="^",
           color="tab:red", s=70,
           label="Ehsan Fig. 2 (0 K nontherm., cubic, zone-boundary)")
ax.scatter(eh_500K["q_sq"], eh_500K["omega_sq_meV2"], marker="v",
           color="tab:orange", s=70,
           label="Ehsan Fig. 2 (500 K therm., cubic, zone-boundary)")

# HAS 1971 cubic-phase soft TO (Wave 2): the FIRST cubic-phase BTO Λ² source.
df_has = loaded["BTO/harada_axe_shirane_1971/cubic_dispersion.csv"]
has_to = df_has[df_has["branch"] == "TO"].copy()
has_to["q_sq"] = has_to["q_inv_angstrom"].astype(float) ** 2
has_to["omega_sq_meV2"] = has_to["omega_meV"].astype(float) ** 2
ax.plot(q_ref**2, 3.0**2 + 972.0 * q_ref**2, color="tab:purple", ls=":", lw=1.3,
        label=r"HAS A(100): $\Omega^2 = 3^2 + 972\,q^2$ (cubic [100], 150 °C)")
ax.scatter(has_to["q_sq"], has_to["omega_sq_meV2"], marker="D",
           color="tab:purple", s=75,
           label="HAS 1971 Fig. 5 soft-TO (cubic, [100], ζ≤0.3)")

ax.set_xlabel(r"$q^2$  (Å$^{-2}$)")
ax.set_ylabel(r"$\Omega^2$  (meV$^2$)")
ax.set_title("Figure B — BTO Λ² source overlay (no fits)")
ax.legend(fontsize=8, loc="upper left")
ax.set_xlim(-0.05, 1.6)
fig.tight_layout()
plt.show()


def _implied_lambda(df_sub):
    rows = df_sub.sort_values("q_sq")
    if len(rows) < 2:
        return None
    g = rows.iloc[0]
    x = rows.iloc[-1]
    if x["q_sq"] - g["q_sq"] == 0:
        return None
    return (x["omega_sq_meV2"] - g["omega_sq_meV2"]) / (x["q_sq"] - g["q_sq"])


lam_0K_implied = _implied_lambda(eh_0K)
lam_500K_implied = _implied_lambda(eh_500K)
print(f"Ehsan (Ω²(X)-Ω²(Γ))/q²(X) — 0 K nontherm.:    {lam_0K_implied:.0f} meV²·Å²")
print(f"Ehsan (Ω²(X)-Ω²(Γ))/q²(X) — 500 K therm.:     {lam_500K_implied:.0f} meV²·Å²")
print("Shirane prior central:                        5100 meV²·Å²")
print()
print("NOTE (v1 fix, option 3):")
print("  Ehsan finite-q points are at zone-boundary (X ≈ 0.78 Å⁻¹), outside")
print("  the small-q regime where Ω²(q) = Ω₀² + Λ²q² holds. The two-point")
print("  'implied Λ²' values above are NOT real Λ² estimates and the negative")
print("  slope at 500 K is consistent with SCP soft-branch anti-crossing with")
print("  another mode along Γ→X, not Λ² < 0. The Ehsan dispersion is dropped")
print("  from the BTO Λ² refit in NB 06; only Shirane Σ + Shirane Δ small-q")
print("  branches feed the refit (both 295 K tetragonal, ±20% envelope")
print("  absorbing the phase-shift uncertainty).")

print()
print("Wave-2: HAS 1971 adds the first CUBIC-phase BTO Λ² source.")
print("  A(100) = 972±50 meV²Å² ([100], soft). The tetragonal Shirane 5100")
print("  is NOT in conflict: HAS A(110)=A(111)=4750 ≈ 5100 — the gap is the")
print("  soft-TO anisotropy (Fig. 7), not a discrepancy.")
record_checklist("Figure B renders (+HAS cubic-phase Λ² source)", True)
record_checklist(
    "Ehsan cubic_dispersion q_inv_angstrom in Å⁻¹",
    True,
    "converted v1 from r.l.u. via 2π/4.013; see extraction_log.md",
)


## 5. Figure C — STO $\Omega_{\rm TO1}(T)$ triangulation

Six curves on one panel, 0–500 K. Tadano 2015 Fig. 5 contributes
`SCPH_with_PM`, `SCPH_without_PM`, `TDEP`, `exp_Servoin`, `exp_YamadaShirane`
for the TO1 mode; Verdi 2023 Fig. 4(a) contributes `MLFF_SSCHA` (with the
4.93 meV downward shift marked in the legend). No fits.

In [ ]:
df_ta = loaded["STO/tadano_2015/scph_omega_T.csv"]
df_vd = loaded["STO/verdi_2023/sscha_omega_T.csv"]

fig, ax = plt.subplots(figsize=(9, 5.5))

tadano_to1 = df_ta[df_ta["mode"] == "TO1"]
style = {
    "SCPH_with_PM":   dict(marker="o", color="tab:blue",  ls="-",  label="Tadano SCPH (+PM)"),
    "SCPH_without_PM":dict(marker="s", color="tab:cyan",  ls="--", label="Tadano SCPH (−PM)"),
    "TDEP":           dict(marker="D", color="tab:green", ls=":",  label="Tadano TDEP"),
    "exp_Servoin":    dict(marker="^", color="tab:red",   ls="",   label="exp Servoin (in Tadano)"),
    "exp_YamadaShirane": dict(marker="v", color="tab:purple", ls="", label="exp Yamada–Shirane (in Tadano)"),
}
for m, st in style.items():
    sub = tadano_to1[tadano_to1["method"] == m].sort_values("T_K")
    ax.plot(sub["T_K"], sub["Omega_meV"], markersize=6, **st)

vd_mlff = df_vd[df_vd["method"] == "MLFF_SSCHA"].sort_values("T_K")
ax.plot(vd_mlff["T_K"], vd_mlff["Omega_meV"], marker="x", color="tab:orange",
        ls="-.", label="Verdi MLFF+SSCHA (shifted −4.93 meV)")

# --- Wave-1 additions: extend the overlay to a 4-source comparison ---
# Fauqué Γ-point TO experimental (INS) extends STO Ω(T) out to 1400 K (raw).
df_fq = loaded["STO/fauque_2022/dho_fits_T.csv"]
fq_to = df_fq[(df_fq["mode"] == "TO") & (df_fq["method"] == "experimental")].sort_values("T_K")
ax.plot(fq_to["T_K"], fq_to["omega_meV"], marker="*", color="black", ls="-",
        markersize=10, label="Fauqué Γ-TO exp (INS, raw)")
# Schmidt ST¹⁶O SSCHA (0–50 K; raw, imaginary Ω²<0 tabulated as negative).
df_sc = loaded["STO/schmidt_2025/sscha_omega_T.csv"]
sc16 = df_sc[(df_sc["isotope"] == "ST16O") & (df_sc["method"] == "SSCHA")].sort_values("T_K")
ax.plot(sc16["T_K"], sc16["Omega_meV"], marker="P", color="tab:brown", ls="-",
        markersize=6, label="Schmidt ST¹⁶O SSCHA (raw)")

# Yamada-Shirane 1969 Table II — gold-standard experimental STO q=0 Ω(T) (Wave 2).
df_ys = loaded["STO/yamada_shirane_1969/omega_T.csv"]
ys_q0 = df_ys[df_ys["condition_direction"] == "q_0_TO_soft"].sort_values("T_K")
ax.plot(ys_q0["T_K"], ys_q0["omega_meV"], marker="o", color="dimgray", ls="-",
        markersize=5, markerfacecolor="none",
        label="Yamada-Shirane 1969 exp (Table II, q=0)")

# Vogt 1995 Fig. 6 cubic_F1u experimental q=0 Ω(T) (Vogt wave; 6th source).
df_vg = loaded["STO/vogt_1995/omega_T.csv"]
vg_f1u = df_vg[df_vg["phase_mode"] == "cubic_F1u"].sort_values("T_K")
ax.plot(vg_f1u["T_K"], vg_f1u["Omega_meV"], marker="D", color="tab:olive", ls="-",
        markersize=5, label="Vogt 1995 hyper-Raman (cubic F1u, q=0)")

ax.set_xlim(0, 1450)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\hbar\Omega_{\rm TO1}$ (meV)")
ax.set_title("Figure C — STO TO1 Ω(T) triangulation, 6 sources (raw, no fits)")
ax.legend(fontsize=8, loc="upper left")
# Each computational source carries its own shift convention; raw values plotted.
ax.text(0.985, 0.03,
        "Shift conventions (raw values plotted):\n"
        "  Verdi MLFF+SSCHA: −4.93 meV (match low-T exp)\n"
        "  Schmidt: imaginary Ω tabulated as negative\n"
        "  Ranalli (not shown): +34.71 cm⁻¹",
        transform=ax.transAxes, fontsize=7, ha="right", va="bottom",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
fig.tight_layout()
plt.show()
record_checklist("Figure C renders (6-source overlay: +Fauqué +Schmidt +YS +Vogt F1u)", True)


## 6. Figure D — STO LST cross-check ($1/\epsilon$ vs $\Omega^2$)

The Lyddane–Sachs–Teller relation says $\Omega^2_{\rm TO} \propto 1/\epsilon$ at
fixed $\epsilon_\infty$, so plotting Verdi's $1/\epsilon(T)$ against Verdi's
$\Omega^2_{\rm TO1}(T)$ on the same $T$ grid (interpolating the dielectric
data onto the SSCHA $T$ points) should be linear. A poor $R^2$ flags an
internal inconsistency in the SSCHA results. **Not a fit panel for kernel
inputs** — purely a consistency check.

In [ ]:
df_eps = loaded["STO/verdi_2023/inverse_dielectric_T.csv"]
df_om = loaded["STO/verdi_2023/sscha_omega_T.csv"]

eps_mlff = df_eps[df_eps["method"] == "MLFF_SSCHA"].sort_values("T_K")
om_mlff = df_om[df_om["method"] == "MLFF_SSCHA"].sort_values("T_K")

# Interpolate 1/eps onto the Omega T grid.
inv_eps_on_om = np.interp(
    om_mlff["T_K"].to_numpy(dtype=float),
    eps_mlff["T_K"].to_numpy(dtype=float),
    eps_mlff["inverse_epsilon"].to_numpy(dtype=float),
)
Omega_sq = om_mlff["Omega_sq_meV2"].to_numpy(dtype=float)
T_vals = om_mlff["T_K"].to_numpy(dtype=float)

# Discard the T<eps_T_min points (where np.interp clamps to endpoint).
mask = T_vals >= eps_mlff["T_K"].min()
x = Omega_sq[mask]
y = inv_eps_on_om[mask]
T_pts = T_vals[mask]

slope, intercept = np.polyfit(x, y, deg=1)
y_pred = slope * x + intercept
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_sq = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x, y, c=T_pts, cmap="viridis", s=60, edgecolor="k", lw=0.4,
                label="Verdi MLFF+SSCHA")
xx = np.linspace(x.min(), x.max(), 100)
ax.plot(xx, slope * xx + intercept, "r--", lw=1.0,
        label=fr"linear fit, $R^2$={r_sq:.4f}")
ax.set_xlabel(r"$\Omega^2_{\rm TO1}$ (meV$^2$, MLFF+SSCHA, shifted)")
ax.set_ylabel(r"$1/\epsilon$")
ax.set_title("Figure D — STO LST cross-check (Verdi)")
cb = fig.colorbar(sc, ax=ax)
cb.set_label("T (K)")
ax.legend(fontsize=9, loc="upper left")
fig.tight_layout()
plt.show()

print(f"Linear fit: 1/eps = {slope:.6f} * Omega^2 + {intercept:.6f}")
print(f"R^2 = {r_sq:.4f}")
record_checklist("Figure D renders", True)
record_checklist("Verdi LST linear-fit R^2 >= 0.9",
                 r_sq >= 0.9, f"R^2 = {r_sq:.4f}")


## 7. Figure E — BTO $T_c$(cubic–tetra) triangulation

Four reported / digitized values plus one chat-2 reference number (Cochran
fit, **not refit here**, kept in the caption).

- Ehsan F-crossing (Fig. 5): ~455 K
- Tadano 2022 raw (Table I): 606 K
- Tadano 2022 lattice-corrected (body text): 511 K
- Experimental (Table I): 390 K
- Cochran two-branch fit from chat 2 (informational only): 437 K

In [ ]:
# Pull Ehsan F-crossing from free_energy_T.csv: T where F_cubic - F_tetra crosses 0.
df_F = loaded["BTO/ehsan_2021/free_energy_T.csv"].copy()
mask = ~df_F["F_tetra_meV"].isna()
sub = df_F[mask].sort_values("T_K")
dF = (sub["F_cubic_meV"].astype(float) - sub["F_tetra_meV"].astype(float)).to_numpy()
T_F = sub["T_K"].astype(float).to_numpy()
# Linear interpolation to the zero crossing.
T_ehsan_cross = float("nan")
for i in range(len(dF) - 1):
    if dF[i] * dF[i + 1] <= 0 and dF[i] != dF[i + 1]:
        T_ehsan_cross = T_F[i] - dF[i] * (T_F[i + 1] - T_F[i]) / (dF[i + 1] - dF[i])
        break

# Tadano values from CSV.
df_tt = loaded["BTO/tadano_2022/transition_temperatures.csv"]
ctt = df_tt[df_tt["transition"] == "cubic-tetra"].iloc[0]
T_tadano_raw = float(ctt["T_c_calc_K"])
T_exp = float(ctt["T_c_exp_K"])
T_tadano_corr = 511.0  # body text
T_cochran_chat2 = 437.0  # informational

labels = [
    f"Ehsan F-cross\n({T_ehsan_cross:.0f} K)",
    f"Tadano raw\n({T_tadano_raw:.0f} K)",
    f"Tadano latt.-corr.\n({T_tadano_corr:.0f} K)",
    f"Experimental\n({T_exp:.0f} K)",
]
vals = [T_ehsan_cross, T_tadano_raw, T_tadano_corr, T_exp]
colors = ["tab:purple", "tab:blue", "tab:cyan", "tab:red"]

fig, ax = plt.subplots(figsize=(8, 4.6))
bars = ax.bar(labels, vals, color=colors, edgecolor="k")
ax.axhline(T_exp, color="tab:red", ls=":", lw=0.9)
ax.axhline(T_cochran_chat2, color="tab:gray", ls="--", lw=0.9,
           label=f"Cochran chat-2 fit (informational): {T_cochran_chat2:.0f} K")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 8, f"{v:.0f} K",
            ha="center", va="bottom", fontsize=9)
ax.set_ylabel(r"$T_c$(cubic-tetra) (K)")
ax.set_title("Figure E — BTO $T_c$ triangulation")
ax.set_ylim(0, max(vals) * 1.15)
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

print(f"Computed Ehsan F-crossing (linear interp on F_cubic - F_tetra): {T_ehsan_cross:.1f} K")
print("Expected value: ~455 K")
record_checklist("Figure E renders", True)
record_checklist("Ehsan F-crossing in 440-470 K window",
                 440 <= T_ehsan_cross <= 470,
                 f"got {T_ehsan_cross:.1f} K")


## 7b. Figure F — KTaO$_3$ $\Omega_{\rm TO}(T)$ (Meier 2022, third material)


In [ ]:
df_kt = loaded["KTO/meier_2022/omega_T.csv"]
df_rn = loaded["KTO/ranalli_2023/omega_T.csv"]
fig, ax = plt.subplots(figsize=(8.5, 5.3))
# Meier 2022 (unshifted)
for m, st in {
    "Calculated":  dict(marker="o", color="tab:blue", ls="-", label="Meier SSCHA+MLIP (calc, unshifted)"),
    "Shirane_exp": dict(marker="s", color="tab:red",  ls="",  label="Meier: Shirane INS (exp)"),
}.items():
    sub = df_kt[df_kt["method"] == m].sort_values("T_K")
    ax.plot(sub["T_K"], sub["Omega_meV"], markersize=6, **st)
# Ranalli 2023 (MLFF+SSCHA shifted -4.30 meV; INS exp)
rn_ml = df_rn[df_rn["method"] == "MLFF+SSCHA"].sort_values("T_K")
rn_ex = df_rn[df_rn["method"] == "expt"].sort_values("T_K")
ax.plot(rn_ml["T_K"], rn_ml["Omega_meV"], marker="^", color="tab:green", ls="--",
        markersize=6, label="Ranalli MLFF+SSCHA (shifted −4.30 meV)")
ax.plot(rn_ex["T_K"], rn_ex["Omega_meV"], marker="v", color="tab:purple", ls="",
        markersize=6, markerfacecolor="none", label="Ranalli INS exp")
# Vogt 1995 Fig. 5 experimental q=0 Ω(T) — the tie-breaker (Vogt wave).
df_vk = loaded["KTO/vogt_1995/omega_T.csv"]
vk = df_vk[df_vk["method"] == "exp_HyperRaman_Vogt1995"].sort_values("T_K")
ax.plot(vk["T_K"], vk["Omega_meV"], marker="D", color="tab:olive", ls="-",
        markersize=5, label="Vogt 1995 hyper-Raman (exp, q=0)")

ax.set_xlim(0, 310)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\hbar\Omega_{\rm TO}$ (meV)")
ax.set_title("Figure F — KTaO₃ TO soft-mode Ω(T): three-source triangulation (raw)")
ax.text(0.02, 0.98,
        "Two computational sources + INS (native cm⁻¹ → meV):\n"
        "  shift conventions — Ranalli −4.30 meV (=34.71 cm⁻¹), Meier unshifted\n"
        "  INS exp anchors agree (~0.4 meV @300 K); computed spread ~1.8 meV\n"
        "  Vogt exp (q=0) tie-breaks: supports Meier-like high-T freq → Ranalli\n"
        "    −4.30 meV downshift slightly over-applied above ~150 K\n"
        "  Λ²_KTO is Γ-point-only in all three sources → bracket, not fit",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=7.5, loc="lower right")
fig.tight_layout()
plt.show()
record_checklist("Figure F (KTaO₃ Ω(T), 3-source: Meier + Ranalli + Vogt exp) renders", True)


## 7b2. Figure F2 — KTaO$_3$ $1/\epsilon(T)$ (Meier + Ranalli; LST input)


In [ ]:
dm = loaded["KTO/meier_2022/inverse_dielectric_T.csv"]
dr = loaded["KTO/ranalli_2023/inverse_dielectric_T.csv"]
fig, ax = plt.subplots(figsize=(8, 5))
for m, st in {
    "Calculated":    dict(marker="o", color="tab:blue", ls="-", label="Meier calc"),
    "Fujishita_exp": dict(marker="s", color="tab:red", ls="", label="Meier: Fujishita exp"),
    "Barrett_fit":   dict(marker="", color="0.5", ls=":", label="Meier: Barrett fit"),
}.items():
    sub = dm[dm["method"] == m].sort_values("T_K")
    if not sub.empty:
        ax.plot(sub["T_K"], sub["inverse_epsilon"], markersize=5, **st)
for m, st in {
    "MLFF+SSCHA":      dict(marker="^", color="tab:green", ls="--", label="Ranalli MLFF+SSCHA"),
    "expt":            dict(marker="v", color="tab:purple", ls="", markerfacecolor="none", label="Ranalli exp"),
    "Curie-Weiss_fit": dict(marker="", color="tab:cyan", ls="-.", label="Ranalli Curie-Weiss"),
}.items():
    sub = dr[dr["method"] == m].sort_values("T_K")
    if not sub.empty:
        ax.plot(sub["T_K"], sub["inverse_epsilon"], markersize=5, **st)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$1/\epsilon$")
ax.set_title("Figure F2 — KTaO₃ 1/ε(T): Meier + Ranalli (LST cross-check input)")
ax.text(0.02, 0.98,
        "Barrett low-T saturation = quantum-paraelectric signature;\n"
        "feeds the KTaO₃ LST 1/ε vs Ω² check (NB 07b, parallel to STO R²=0.998)",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()
plt.show()
record_checklist("Figure F2 (KTaO₃ 1/ε(T), Meier + Ranalli) renders", True)


## 7c. Figure G — STO soft-mode $\Gamma(T)$ (Yamada-Shirane 1969 Fig. 6)


In [ ]:
df_ysg = loaded["STO/yamada_shirane_1969/gamma_T.csv"].sort_values("T_K")
fig, ax = plt.subplots(figsize=(7.5, 4.8))
yerr = df_ysg["gamma_uncertainty_meV"].astype(float).replace(0, np.nan)
ax.errorbar(df_ysg["T_K"], df_ysg["gamma_meV"], yerr=yerr, marker="o",
            color="tab:red", ls="-", capsize=3, label="YS 1969 Fig. 6 (q=0, INS)")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\hbar\Gamma$ (meV)")
ax.set_title("Figure G — STO soft-mode Gamma(T) (Yamada-Shirane 1969 Fig. 6)")
ax.text(0.03, 0.97,
        "canonical low-T exp STO linewidth\n0.24 -> 0.84 meV (20-250 K)\n"
        "underdamped: eta = Gamma/Omega ~ 0.09 @250 K\nenables eta_fit alongside eta_bracket (NB 08)",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
record_checklist("Figure G (STO Gamma(T), YS 1969) renders", True)


## 7d. Figure H — cubic BTO soft-TO + TA dispersion (Harada-Axe-Shirane 1971 Fig. 5)


In [ ]:
df_h = loaded["BTO/harada_axe_shirane_1971/cubic_dispersion.csv"]
fig, ax = plt.subplots(figsize=(8, 5))
for br, st in {
    "TO": dict(marker="D", color="tab:purple", label="soft TO (quasiharmonic Omega)"),
    "TA": dict(marker="o", color="tab:green", label="TA (observed)"),
}.items():
    sub = df_h[df_h["branch"] == br].sort_values("q_inv_angstrom")
    ax.plot(sub["q_inv_angstrom"].astype(float), sub["omega_meV"].astype(float),
            ls="", markersize=7, **st)
qq = np.linspace(0, 0.5, 100)
ax.plot(qq, np.sqrt(3.0**2 + 972.0 * qq**2), "k--", lw=1.1,
        label=r"$\Omega=\sqrt{3^2+972\,q^2}$  (cubic Lambda^2 fit, [100])")
ax.set_xlabel(r"$q$ along [100]  (Å$^{-1}$)")
ax.set_ylabel(r"$\hbar\omega$  (meV)")
ax.set_title("Figure H — cubic BTO soft-TO + TA dispersion (HAS 1971 Fig. 5, 150 C)")
ax.text(0.03, 0.97,
        "first CUBIC-phase BTO source (removes the tetragonal caveat)\n"
        "soft TO overdamped (Gamma ~ 2.3 Omega) -> points are quasiharmonic Omega",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()
record_checklist("Figure H (cubic BTO dispersion, HAS 1971) renders", True)


## 7e. Figure I — STO soft-mode damping: YS 1969 vs Vogt 1995 (HWHM basis)


In [ ]:
ys = loaded["STO/yamada_shirane_1969/gamma_T.csv"].sort_values("T_K")
vg = loaded["STO/vogt_1995/gamma_T.csv"]
vg_c = vg[vg["phase_mode"] == "cubic_F1u"].sort_values("T_K")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ys["T_K"], ys["gamma_meV"], marker="o", color="tab:blue", ls="-",
        label="YS 1969 (neutron, HWHM direct)")
ax.plot(vg_c["T_K"], vg_c["gamma_meV"], marker="s", color="tab:red", ls="--",
        label="Vogt 1995 plotted γ (FWHM)")
ax.plot(vg_c["T_K"], vg_c["gamma_meV"] / 2, marker="D", color="tab:green", ls="-",
        label="Vogt Γ_HWHM = γ/2 (canonical basis)")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\Gamma$ (meV)")
ax.set_title("Figure I — STO soft-mode damping: YS 1969 vs Vogt 1995 (HWHM basis)")
ax.text(0.02, 0.97,
        "YS: neutron, finite-q-analyzed, central-peak-contaminated\n"
        "Vogt: hyper-Raman true q=0, central peak (δ) separated\n"
        "raw γ ratio ~3.3×; ~1.7× residual after FWHM→HWHM /2\n"
        "documented physics (q-res + central peak), NOT an error",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()
record_checklist("Figure I (YS vs Vogt STO damping, HWHM basis) renders", True)


## 7f. Figure J — η preview: STO ≈ KTaO$_3$ cluster (underdamped), BTO alone


In [ ]:
def _eta_curve(dfo, dfg, ometh=None):
    o = dfo[dfo["method"] == ometh] if ometh else dfo
    o = o.sort_values("T_K"); g = dfg.sort_values("T_K")
    gT = g["T_K"].to_numpy(float); gV = g["gamma_meV"].to_numpy(float)
    oT = o["T_K"].to_numpy(float); oV = o["Omega_meV"].to_numpy(float)
    m = (oT >= gT.min()) & (oT <= gT.max())
    oT, oV = oT[m], oV[m]
    return oT, 0.5 * np.interp(oT, gT, gV) / oV   # FWHM->HWHM (gamma/2)

so = loaded["STO/vogt_1995/omega_T.csv"]; so = so[so["phase_mode"] == "cubic_F1u"]
sg = loaded["STO/vogt_1995/gamma_T.csv"]; sg = sg[sg["phase_mode"] == "cubic_F1u"]
ko = loaded["KTO/vogt_1995/omega_T.csv"]; kg = loaded["KTO/vogt_1995/gamma_T.csv"]
sT, sE = _eta_curve(so, sg)
kT, kE = _eta_curve(ko, kg, ometh="exp_HyperRaman_Vogt1995")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sT, sE, marker="o", color="tab:blue", label="STO (Vogt cubic F1u)")
ax.plot(kT, kE, marker="s", color="tab:green", label="KTaO₃ (Vogt)")
ax.axhline(0.5, color="0.5", ls="--", label="η = 0.5 (under/over-damped)")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel(r"$\eta = \Gamma_{\rm HWHM}/\Omega$")
ax.set_ylim(0, 0.6)
ax.set_title("Figure J — η preview: STO ≈ KTaO₃ cluster (underdamped); BTO η≈2.3 off-scale")
ax.text(0.02, 0.96,
        "Two QPEs CLUSTER at η≈0.15 (FWHM basis); empty interior up to BTO η≈2.3.\n"
        "Factor-of-2 convention (γ as HWHM → η≈0.29) cancels in the STO–KTaO₃ comparison.",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()
record_checklist("Figure J (η preview cluster: STO≈KTaO₃ underdamped) renders", True)


## 8. STO $\Gamma$ availability summary

Every CSV column / row that carries an STO soft-mode linewidth or lifetime,
with $T$ and source. This must make clear that we currently have
one anchor (Tadano 2015 Fig. 7 inset at 300 K) and that Yamada–Shirane Γ(T) is
still outstanding.

In [ ]:
gamma_sources: list[dict] = []

df = loaded.get("STO/yamada_shirane_1969/gamma_T.csv")
if df is not None and not df.empty:
    for _, r in df.iterrows():
        gamma_sources.append({
            "source": "Yamada-Shirane 1969 Fig. 6",
            "T_K": float(r["T_K"]),
            "value": float(r["gamma_meV"]),
            "kind": "Gamma_meV (intrinsic width, neutron INS) [Wave-2 VERIFIED vs Fig. 6]",
        })

df = loaded.get("STO/fauque_2022/dho_fits_T.csv")
fq_no_gamma = df is not None and df["gamma_meV"].isna().all()
gamma_sources.append({
    "source": "Fauqué 2022 Fig. 1(d)",
    "T_K": "4–1400",
    "value": "NONE (ω only; gamma_meV=NaN)" if fq_no_gamma else "UNEXPECTED Γ present!",
    "kind": "DHO Γ NOT reported (SI checked) — η stays a bracket for STO",
})
gamma_sources.append({
    "source": "Tadano 2015 Fig. 7 inset",
    "T_K": 300,
    "value": "anchor (not digitized into any CSV)",
    "kind": "phonon lifetime / linewidth (referenced in extraction_log.md)",
})
ir_catalog = DATA_RAW / "STO_ir_data.json"
if ir_catalog.exists():
    d = json.loads(ir_catalog.read_text())
    peak = min(d["peaks"], key=lambda p: p["frequency_cm1"])
    gamma_sources.append({
        "source": "STO IR catalog (v1 proxy)",
        "T_K": "polynomial",
        "value": f"A_k={peak['A_k']}, B_k={peak['B_k']}, C_k={peak['C_k']} (cm^-1)",
        "kind": "Gamma(T) = A + B*T + C*T^2 (catalog Lorentz parameterization)",
    })

# Vogt 1995 hyper-Raman γ(T) — NEW PRIMARY η source for STO (and KTaO3).
for _mat, _path, _filt in [("STO", "STO/vogt_1995/gamma_T.csv", "cubic_F1u"),
                           ("KTaO3", "KTO/vogt_1995/gamma_T.csv", None)]:
    dvg = loaded.get(_path)
    if dvg is not None and not dvg.empty:
        sub = dvg[dvg["phase_mode"] == _filt] if (_filt and "phase_mode" in dvg.columns) else dvg
        hi = sub.sort_values("T_K").iloc[-1]
        gamma_sources.append({
            "source": f"Vogt 1995 {_mat} (HR)",
            "T_K": float(hi["T_K"]),
            "value": f"gamma={hi['gamma_meV']} meV (plotted FWHM -> HWHM=gamma/2)",
            "kind": "PRIMARY eta (true q=0, central peak separated) -> eta_fit",
        })

print(f"{'Source':32s}  {'T (K)':>12}  {'Value':40s}  Kind")
print("-" * 110)
for g in gamma_sources:
    print(f"{g['source']:32s}  {str(g['T_K']):>12s}  {str(g['value']):40s}  {g['kind']}")

mp = loaded["BTO/harada_axe_shirane_1971/mode_parameters.csv"]
to = mp[mp["mode"] == "TO"]
eta_bto = float((to["gamma_meV"] / to["omega_meV"]).mean())
ysg = loaded["STO/yamada_shirane_1969/gamma_T.csv"]
yso = loaded["STO/yamada_shirane_1969/omega_T.csv"]
yso = yso[yso["condition_direction"] == "q_0_TO_soft"]
g250 = float(ysg[ysg["T_K"] == 250]["gamma_meV"].iloc[0])
o250 = float(yso[yso["T_K"] == 250]["omega_meV"].iloc[0])
eta_sto = g250 / o250
print()
print(f"DAMPING REGIME (verified): cubic BTO η=Γ/Ω ≈ {eta_bto:.2f} (HAS 1971, overdamped Γ≳2Ω)")
print(f"                           cubic STO η=Γ/Ω ≈ {eta_sto:.3f} @250K (YS 1969, underdamped)")
print()
print("Bottom line: STO Γ(T) is solved by TWO experiments — Yamada-Shirane 1969 (neutron,")
print("HWHM, finite-q, central-peak-contaminated) and Vogt 1995 hyper-Raman (true q=0,")
print("central peak separated). VOGT IS THE PRIMARY η SOURCE; YS is a documented cross-")
print("check. They disagree ~3.3× raw (~1.7× after the γ FWHM→HWHM /2) — documented")
print("physics (q-resolution + central peak), NOT a data error. KTaO3 Γ(T) likewise solved")
print("by Vogt Fig. 5 (case 'a' → eta_fit). γ-convention: plotted Vogt γ = FWHM →")
print("Γ_HWHM = γ/2 (factor-of-2 noted; Refs 29-30 would make airtight). HAS 1971 keeps")
print("cubic BTO overdamped (η≈2.3); Fauqué reports no Γ. KTaO3 Vogt γ tail RE-DIGITIZED")
print("and verified vs Fig. 5 (295 K = 21 cm⁻¹ = 2.60 meV, within ~1 cm⁻¹); 140-170 K is")
print("smoothed (figure scatters ~±1 cm⁻¹, sub-error-bar). Sub-60 K γ/Ω carry ±0.17 meV")
print("(stat 0.12 ⊕ sys 0.12 quadrature; Vogt defect-induced sub-60 K systematic, p.8051),")
print("consistently across STO and KTaO3 Vogt files; STO A2u/Eu share one γ (Flag-2 fixed).")
record_checklist("STO+KTaO3 Γ(T) documented (Vogt primary; YS cross-check; γ-conv recorded)",
                 fq_no_gamma and not ysg.empty, f"{len(gamma_sources)} sources logged")


## 9. BTO $\Lambda^2$ data point phase tags

Every BTO $\Lambda^2$ data point with $T$ and phase. The cubic-phase kernel
needs cubic-phase $\Lambda^2$, so the predominance of 295 K tetragonal Shirane
data here is a real caveat.

In [ ]:
print(f"{'Source':45s}  {'T (K)':>7}  {'Phase':12s}  Notes")
print("-" * 105)

# Shirane prior fit (Lambda_sq.csv)
df = loaded["BTO/shirane_1970/Lambda_sq.csv"]
for _, r in df.iterrows():
    print(f"{'Shirane 1970 prior fit (chat 2)':45s}  "
          f"{int(r['T_K']):>7d}  {'tetragonal':12s}  "
          f"Lambda^2={r['Lambda_sq_meV2_A2']} meV^2 A^2; ref range "
          f"{r['reference_q_range']}")

# Shirane Σ branch (Fig 3 TO points) and Δ branch (Fig 6 TO points)
df = loaded["BTO/shirane_1970/cubic_dispersion.csv"]
to_pts = df[df["branch"] == "TO"]
for fig_label in ["Fig_3", "Fig_6"]:
    sub = to_pts[to_pts["source_figure"] == fig_label]
    if not sub.empty:
        n = len(sub)
        T = int(sub["T_K"].iloc[0])
        phase = sub["phase"].iloc[0]
        pol = sub["polarization_e"].iloc[0]
        wv = sub["wavevector_q"].iloc[0]
        print(f"{'Shirane 1970 ' + fig_label + ' TO branch':45s}  "
              f"{T:>7d}  {phase:12s}  "
              f"{n} pts; polarization_e={pol}, wavevector_q={wv}")

# Ehsan finite-q soft-mode points
df = loaded["BTO/ehsan_2021/cubic_dispersion.csv"]
soft = df[df["branch_label"] == "Ferroelectric_Soft_Mode"]
for cond in ["nonthermalized", "thermalized"]:
    sub = soft[soft["condition"] == cond]
    if not sub.empty:
        T = int(sub["T_K"].iloc[0])
        phase = "cubic (ref. struct.)" if cond == "nonthermalized" else "cubic"
        print(f"{'Ehsan 2021 Fig. 2 soft-mode (' + cond + ')':45s}  "
              f"{T:>7d}  {phase:12s}  "
              f"{len(sub)} q-points incl. Γ")

# HAS 1971 cubic-phase source (Wave 2): resolves the all-tetragonal caveat below.
df = loaded["BTO/harada_axe_shirane_1971/mode_parameters.csv"]
to = df[df["mode"] == "TO"]
for T in sorted(to["T_K"].unique()):
    sub = to[to["T_K"] == T]
    print(f"{'HAS 1971 Table I soft-TO (Omega,Gamma)':45s}  {int(T):>7d}  {'cubic':12s}  "
          f"{len(sub)} pts; Gamma/Omega~{(sub['gamma_meV']/sub['omega_meV']).mean():.1f} (overdamped)")
ls = loaded["BTO/harada_axe_shirane_1971/Lambda_sq.csv"].iloc[0]
print(f"{'HAS 1971 cubic Lambda^2(100) [Fig. 5]':45s}  {int(ls['T_K']):>7d}  {'cubic':12s}  "
      f"Lambda^2={ls['Lambda_sq_meV2_A2']}+/-{ls['Lambda_sq_uncertainty_meV2_A2']} meV^2 A^2 (small-q)")

print()
print("Caveat (UPDATED Wave 2): Shirane 1970 (Sigma,Delta) + the Shirane prior fit are")
print("295 K TETRAGONAL; Ehsan Fig. 2 is cubic but zone-boundary only. HAS 1971 now")
print("supplies the first CUBIC-phase small-q BTO Lambda^2 (A(100)=972+/-50 meV^2 A^2 at")
print("150 C), removing the all-tetragonal limitation. Note the strong soft-TO anisotropy")
print("(A(100)=972 vs A(110)=A(111)=4750 ~ Shirane's 5100): the cubic [100] Lambda^2 is")
print("~5x softer than the steep directions, so 'Lambda^2_BTO' must specify a direction in NB 06.")
record_checklist("BTO Lambda^2 phase tags printed (+HAS cubic-phase source)", True)


## 10. Pre-flight checklist

Each item below is either ✓ (passed) or ✗ (failed). The checklist requires *every*
item to be ✓ before v1 module work begins. If any item is ✗, this notebook
must be re-run after the underlying issue is fixed, **before** building
`literature_io.py`, `eta.py`, or `kappa_kernel.py`.

In [ ]:
def _mark(ok: bool) -> str:
    return "✓" if ok else "✗"

print(f"\n{'Item':70s}  {'OK':3s}  Note")
print("-" * 120)
for item, ok, note in CHECKLIST:
    print(f"{item:70s}  {_mark(ok):3s}  {note}")

print()
n_pass = sum(1 for _, ok, _ in CHECKLIST if ok)
n_total = len(CHECKLIST)
all_ok = n_pass == n_total
print(f"{n_pass}/{n_total} checklist items pass.")
if all_ok:
    print("\nPre-flight GATE: PASS. NB 05b complete; "
          "proceed to literature_io.py.")
else:
    failing = [(i, n) for i, ok, n in CHECKLIST if not ok]
    print("\nPre-flight GATE: BLOCKED. Items requiring resolution:")
    for i, n in failing:
        print(f"  - {i}  ({n})")
    print("\nDo NOT start module work until these items are addressed.")
